# AnnotationAgent и human-in-the-loop

Цель — проверить production-артефакты авторазметки и подготовить честную передачу человеку. `source_label`, `auto_label`, `human_label` и `final_label` намеренно разделены; notebook не создаёт фиктивную ручную разметку.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'config.yaml').exists() and (candidate / 'agents').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.annotation_agent import AnnotationAgent
from agents.common import load_frame

annotator = AnnotationAgent(modality='text', config=ROOT / 'config.yaml')

## Авторазметка production backend

Тяжёлая инференс-операция по умолчанию не запускается из notebook. Её воспроизводимо выполняет первый CLI-запуск. Если нужно экспериментировать интерактивно, переключите `RUN_AUTO_LABEL`; результат в этой ячейке не объявляется human label и не заменяет основной артефакт.

In [ ]:
RUN_AUTO_LABEL = False
clean_path = ROOT / 'data/processed/reviews_clean.parquet'

if RUN_AUTO_LABEL and clean_path.exists():
    preview_labeled = annotator.auto_label(load_frame(clean_path))
    display(preview_labeled[['record_id', 'text', 'source_label', 'auto_label', 'confidence']].head())
elif RUN_AUTO_LABEL:
    print(f'Clean dataset отсутствует: {clean_path}')
else:
    print('Auto-label is disabled; reading the pipeline artifact below.')

In [ ]:
candidates = [
    ROOT / 'data/labeled/reviews_auto_labeled.parquet',
    ROOT / 'data/labeled/reviews_auto_labeled.csv',
]
auto_path = next((path for path in candidates if path.exists()), None)
if auto_path is None:
    labeled = None
    print('Auto-labeled artifact отсутствует. Запустите первый проход pipeline.')
else:
    labeled = load_frame(auto_path)
    metrics = annotator.check_quality(labeled)
    queue = annotator.flag_for_review(labeled)
    print(f'Artifact: {auto_path} · rows={len(labeled)} · review queue={len(queue)}')
    display(metrics)
    display(queue[['record_id', 'text', 'source', 'auto_label', 'confidence', 'review_reason']].head(10))

## Интерпретация качества

Cohen's κ/agreement с `source_label` измеряют согласие модели с weak/gold меткой источника, а не согласие двух людей. Отдельная метрика против `human_label` становится содержательной только после настоящей проверки. Review queue выбирается только по confidence и не показывает `source_label`, чтобы не подсказывать человеку ответ.

In [ ]:
spec_path = ROOT / 'reports/annotation_spec.md'
if spec_path.exists():
    display(Markdown(spec_path.read_text(encoding='utf-8')))
else:
    print('Annotation specification пока не создана; её генерирует первый pipeline run.')

labelstudio_path = ROOT / 'data/review/labelstudio_import.json'
if labelstudio_path.exists():
    tasks = json.loads(labelstudio_path.read_text(encoding='utf-8'))
    print(f'Label Studio export: root={type(tasks).__name__}, tasks={len(tasks)}')
    display(tasks[0] if tasks else {})
else:
    print('Label Studio JSON пока отсутствует.')

## Обязательная ручная точка

1. Выполнить `python run_pipeline.py --review-mode required --force`; процесс должен остановиться со статусом `review_required`.
2. Запустить `python -m streamlit run hitl_app.py`.
3. Прочитать каждую строку, указать имя; пустой `human_label` явно подтверждает auto label, выбранная метка исправляет его.
4. Сохранить `review_queue_corrected.csv` и повторить `python run_pipeline.py --review-mode required`.

Только после валидации reviewer/record IDs второй проход формирует verified HITL. Этот notebook намеренно не заполняет reviewer автоматически.